# NMD Rate Model: Theory

This notebook lays out the mathematical foundation for the two-stage model used to forecast
non-maturing deposit (NMD) rates.

---

## 1. What are NMD rates?

Non-maturing deposits (current accounts, savings accounts) have no contractual maturity.
The bank sets the deposit rate administratively, but it co-moves with market rates, partly and with a lag.
This **pass-through** is heterogeneous across customer segments:
corporate clients are far more rate-sensitive than retail current-account holders.

Modelling NMD rates well matters for:

- **IRRBB** (Interest Rate Risk in the Banking Book): pricing and hedging of implicit optionality.
- **Net Interest Income (NII)** simulation under stress scenarios.
- **Behavioural modelling** for liquidity and ALM purposes.

---

## 2. Model Overview

The model has two stages:

1. **Market rates**: extract latent yield-curve factors from observed swap/government-bond yields
   using the Diebold-Li state-space model.

2. **Deposit repricing**: regress deposit rates on those factors with a Bayesian hierarchical model
   that shares information across segments.

---

## 3. Stage 1: Diebold-Li State-Space Model

### 3.1 Nelson-Siegel yield curve

The Nelson-Siegel (1987) functional form parametrises the yield curve at time $t$ as

$$
y_t(\tau) = L_t + S_t \cdot \ell_1(\tau) + C_t \cdot \ell_2(\tau)
$$

where $\tau$ is the maturity (in months) and the factor loadings are

$$
\ell_1(\tau) = \frac{1 - e^{-\lambda\tau}}{\lambda\tau}, \qquad
\ell_2(\tau) = \frac{1 - e^{-\lambda\tau}}{\lambda\tau} - e^{-\lambda\tau}.
$$

The three factors have natural interpretations:

| Factor | Loading at $\tau \to 0$ | Loading at $\tau \to \infty$ | Interpretation |
|--------|------------------------|------------------------------|----------------|
| $L_t$ | 1 | 1 | **Level**: parallel shifts |
| $S_t$ | 1 | 0 | **Slope**: short end minus long end |
| $C_t$ | 0 | 0 | **Curvature**: hump shape. Reaches it highest in medium-term maturities |

The decay parameter $\lambda$ controls where $\ell_2(\tau)$ reaches its maximum.
We fix $\lambda = 0.0609$ (standard choice for monthly data, peak at ~30 months).

### 3.2 State-space form (Diebold & Li, 2006)

Diebold and Li treat the NS factors as latent states that evolve over time.

**Observation equation** (yields at $J$ maturities):

$$
\mathbf{y}_t = \boldsymbol{\Lambda} \mathbf{f}_t + \boldsymbol{\varepsilon}_t,
\qquad \boldsymbol{\varepsilon}_t \sim \mathcal{N}(\mathbf{0},\, \sigma^2_{\text{obs}} \mathbf{I}_J)
$$

where $\mathbf{f}_t = (L_t, S_t, C_t)^\top$ and $\boldsymbol{\Lambda}$ is the $(J \times 3)$ loading matrix
with rows $[1,\; \ell_1(\tau_j),\; \ell_2(\tau_j)]$.

**State equation** (VAR(1) factor dynamics):

$$
\mathbf{f}_t = \boldsymbol{\mu} + \boldsymbol{\Phi}(\mathbf{f}_{t-1} - \boldsymbol{\mu}) + \boldsymbol{\eta}_t,
\qquad \boldsymbol{\eta}_t \sim \mathcal{N}(\mathbf{0},\, \mathbf{Q})
$$

We use a diagonal $\boldsymbol{\Phi}$ for identification, so each factor mean-reverts independently.

### 3.3 Kalman filter

Because both equations are linear and Gaussian, the **Kalman filter** delivers the exact
marginal likelihood $p(\mathbf{y}_1, \ldots, \mathbf{y}_T \mid \boldsymbol{\theta})$
via the prediction-error decomposition:

$$
\log p(\mathbf{Y} \mid \boldsymbol{\theta}) =
\sum_{t=1}^{T} \left[
-\frac{J}{2}\log(2\pi)
-\frac{1}{2}\log|\mathbf{S}_t|
-\frac{1}{2}\mathbf{v}_t^\top \mathbf{S}_t^{-1} \mathbf{v}_t
\right]
$$

where $\mathbf{v}_t = \mathbf{y}_t - \boldsymbol{\Lambda}\hat{\mathbf{f}}_{t|t-1}$ is the
innovation and $\mathbf{S}_t = \boldsymbol{\Lambda} \mathbf{P}_{t|t-1} \boldsymbol{\Lambda}^\top + \sigma^2_{\text{obs}}\mathbf{I}$
its covariance.

The filter recursions are:

$$
\hat{\mathbf{f}}_{t|t-1} = \boldsymbol{\mu} + \boldsymbol{\Phi}(\hat{\mathbf{f}}_{t-1|t-1} - \boldsymbol{\mu})
\qquad
\mathbf{P}_{t|t-1} = \boldsymbol{\Phi}\mathbf{P}_{t-1|t-1}\boldsymbol{\Phi}^\top + \mathbf{Q}
$$

$$
\mathbf{K}_t = \mathbf{P}_{t|t-1}\boldsymbol{\Lambda}^\top \mathbf{S}_t^{-1}
$$

$$
\hat{\mathbf{f}}_{t|t} = \hat{\mathbf{f}}_{t|t-1} + \mathbf{K}_t \mathbf{v}_t
\qquad
\mathbf{P}_{t|t} = (\mathbf{I} - \mathbf{K}_t \boldsymbol{\Lambda})\mathbf{P}_{t|t-1}
$$

This is implemented via `pytensor.scan` so that PyMC/NUTS can differentiate through the filter.

---

## 4. Stage 2: Hierarchical Deposit Repricing

### 4.1 Pass-through regression

Given the filtered factors $\hat{L}_t, \hat{S}_t, \hat{C}_t$, the deposit rate for
segment $s$ at time $t$ is modelled as a static linear regression:

$$
r_{s,t} = \alpha_s + \beta^L_s \hat{L}_t + \beta^S_s \hat{S}_t + \beta^C_s \hat{C}_t + \varepsilon_{s,t},
\qquad \varepsilon_{s,t} \sim \mathcal{N}(0, \sigma_s^2)
$$

The interpretation of each coefficient:

- $\alpha_s$: segment-specific intercept (floor rate, administered spread).
- $\beta^L_s$: sensitivity to parallel yield curve shifts (main pass-through driver).
- $\beta^S_s$: sensitivity to slope changes (short rate vs. long rate composition effects).
- $\beta^C_s$: sensitivity to curvature (less economically significant, included for completeness).

### 4.2 Hierarchical structure

Each segment gets its own set of coefficients, but these are not estimated independently.
Instead, they are treated as draws from a common population distribution — **partial pooling**:

$$
\alpha_s \sim \mathcal{N}(\mu_\alpha, \sigma_\alpha), \qquad
\beta^L_s \sim \mathcal{N}(\mu_{\beta L}, \sigma_{\beta L}), \quad \ldots
$$

with hyperpriors on $\mu$ and $\sigma$ for each coefficient.

**Why hierarchical?**  
Partial pooling shrinks noisy segment estimates towards the population mean.
Segments with sparse or volatile data borrow strength from the group;
segments with rich data retain their individual estimates.
This is the Bayesian analogue of mixed-effects models.

### 4.3 Non-centred parameterisation

To improve MCMC geometry (avoid funnel-shaped posteriors), we use the
non-centred form:

$$
\beta^L_s = \mu_{\beta L} + \sigma_{\beta L} \cdot z^L_s, \qquad z^L_s \sim \mathcal{N}(0, 1)
$$

and similarly for the other coefficients. The NUTS sampler explores the $z$ space
(standard normal) rather than the original $\beta$ space, which is more efficient
when the group variance $\sigma$ is small.

---

## 5. Segments

We model four NMD segments ordered by increasing rate sensitivity:

| # | Segment | Description | Expected $\beta^L$ |
|---|---------|-------------|--------------------|
| 1 | Retail Current | Transactional accounts; operationally sticky | Low (~0.20) |
| 2 | Retail Savings | Savings products; some competitive pressure | Moderate (~0.35) |
| 3 | SME Operational | Business current accounts; price-aware clients | Moderate-high (~0.50) |
| 4 | Corporate | Treasury-managed; closely tracks market rates | High (~0.75) |

The hierarchical model pools information across these four segments,
capturing the systematic pattern that larger and more financially sophisticated
clients command higher pass-through.

---

## 6. Project Structure

```
NMD rates/
├── src/
│   ├── data.py          — ECB yield curve download (SDW REST API)
│   ├── simulate.py      — DGPs: VAR(1) factors, static pass-through, ECM regime-switching
│   ├── diebold_li.py    — Kalman filter (pytensor) + PyMC state-space model
│   ├── repricing.py     — static hierarchical repricing model
│   └── ecm_repricing.py — ECM + Hamilton filter + PyMC model
├── notebooks/
│   ├── 01_theory.ipynb      — this notebook
│   ├── 02_data.ipynb        — ECB data download + ECM simulation
│   ├── 03_diebold_li.ipynb  — DL estimation on ECB data
│   ├── 04_repricing.ipynb   — ECM regime-switching estimation
│   └── 05_pipeline.ipynb    — end-to-end scenario analysis (ECM)
└── requirements.txt
```

---

## 7. ECM (Error Correction Model)

### 7.1 Motivation

The static pass-through model assumes deposit rates adjust **instantaneously** to changes
in market factors. In practice, banks adjust rates administratively, with delays:

- Competitive pressure is observed with a lag.
- Rate cards are revised periodically, not continuously.
- In low-rate environments, banks may not pass through further cuts at all (floor behaviour).

The **Error Correction Model (ECM)** captures this inertia explicitly.

### 7.2 ECM Formulation

The deposit rate change for segment $s$ at time $t$ is:

$$
\Delta r_{s,t} = \gamma_s \cdot \underbrace{(r_{s,t-1} - r^*_{s,t-1})}_{\text{ECT}_{s,t-1}} + \varepsilon_{s,t}
$$

where:

- $\Delta r_{s,t} = r_{s,t} - r_{s,t-1}$ is the monthly change in deposit rate.
- $r^*_{s,t-1} = \alpha_s + \beta^L_s L_{t-1} + \beta^S_s S_{t-1} + \beta^C_s C_{t-1}$ is the **long-run equilibrium** rate.
- $\text{ECT}_{s,t-1} = r_{s,t-1} - r^*_{s,t-1}$ is the **error correction term** (deviation from equilibrium).
- $\gamma_s \in (-1, 0)$ is the **adjustment speed**.

### 7.3 Interpretation of gamma

| Value of $\gamma_s$ | Behaviour |
|---------------------|-----------|
| $\gamma_s = -1$ | Complete correction in one period (instant adjustment) |
| $\gamma_s = -0.3$ | 30% of the gap closed each month (moderate stickiness) |
| $\gamma_s \to 0$ | Very slow adjustment (extreme stickiness) |

The **half-life** of adjustment is the number of months needed to close half the initial gap:

$$
\tau_{1/2} = \frac{-\log 2}{\log(1 + \gamma_s)} \text{ months}
$$

For example, $\gamma_s = -0.25$ implies a half-life of $\approx 2.4$ months;
$\gamma_s = -0.10$ implies $\approx 6.6$ months.

### 7.4 Hierarchical ECM

We estimate $\gamma_s$ hierarchically with a log-normal prior on $-\gamma_s$
(guaranteeing $\gamma_s < 0$):

$$
\gamma_s = -\exp(\mu_{\log} + \sigma_{\log} \cdot z_s), \qquad z_s \sim \mathcal{N}(0, 1)
$$

---

## 8. Regime-Switching Pass-Through

### 8.1 Motivation

ECB data reveals a striking structural break:

- **2015–2021 (low-rate environment):** deposit rates were at the zero lower bound; further cuts in market
  rates were not passed on. Effective $\beta^L_s \approx 0$.
- **2022–2024 (hiking cycle):** rates rose rapidly; banks passed through significant fractions of
  ECB hikes. Effective $\beta^L_s \approx 0.3$–$0.75$ depending on segment.

A single static $\beta^L_s$ cannot capture this: an in-sample average would understate
pass-through during the hiking cycle and overstate it during the low-rate period.

### 8.2 Hidden Markov Regime Process

We introduce a latent binary regime $z_t \in \{0, 1\}$ following a first-order Markov chain:

$$
P(z_t = j \mid z_{t-1} = i) = P_{ij}
$$

with transition matrix:

$$
P = \begin{pmatrix} p_{00} & 1-p_{00} \\ 1-p_{11} & p_{11} \end{pmatrix}
$$

where $p_{00}$ is the probability of staying in regime 0 (low rate)
and $p_{11}$ the probability of staying in regime 1 (hiking).

The **expected duration** in regime $k$ is $1/(1 - p_{kk})$ periods.

### 8.3 Regime-Specific Pass-Through

The long-run equilibrium becomes:

$$
r^*_{s,t-1}(z_t) = \alpha_s + \beta^{L,(z_t)}_s L_{t-1} + \beta^{S,(z_t)}_s S_{t-1} + \beta^C_s C_{t-1}
$$

where $\beta^{L,(k)}_s$ and $\beta^{S,(k)}_s$ differ by regime.
Curvature sensitivity $\beta^C_s$ is regime-invariant (smaller and harder to identify).

The full ECM with regime switching is:

$$
\Delta r_{s,t} = \gamma_s \cdot (r_{s,t-1} - r^*_{s,t-1}(z_t)) + \varepsilon_{s,t},
\qquad \varepsilon_{s,t} \sim \mathcal{N}(0, \sigma_s^2)
$$

**Identification constraint:** $\mu_{\beta L}^{(0)} < \mu_{\beta L}^{(1)}$, i.e. the population-level
pass-through in regime 1 is strictly higher. This is enforced by:

$$
\mu_{\beta L}^{(1)} = \mu_{\beta L}^{(0)} + \delta, \qquad \delta \sim \text{HalfNormal}(0.30)
$$

### 8.4 Hamilton Filter

Because $z_t$ is latent, we cannot condition on it directly. The **Hamilton (1989) filter**
marginalises over all possible regime paths, analogously to the Kalman filter for continuous states.

**Predict step:** The predicted regime probability is

$$
\xi_{t|t-1} = P^\top \xi_{t-1|t-1}
$$

**Update step:** Given the observed $\Delta r_t$, the likelihood under each regime is

$$
f_k(\Delta r_t) = \prod_s \mathcal{N}(\Delta r_{s,t}; \gamma_s \text{ECT}^{(k)}_{s,t-1}, \sigma_s^2)
$$

and the filtered probability is:

$$
\xi_{t|t}(k) = \frac{\xi_{t|t-1}(k) \cdot f_k(\Delta r_t)}{\sum_j \xi_{t|t-1}(j) \cdot f_j(\Delta r_t)}
$$

The normalisation constant in the denominator is the one-step-ahead likelihood contribution:

$$
p(\Delta r_t \mid \Delta r_{1:t-1}, \boldsymbol{\theta}) = \sum_j \xi_{t|t-1}(j) \cdot f_j(\Delta r_t)
$$

Summing over $t$ gives the **marginal log-likelihood** used as a `pm.Potential` for NUTS.
The entire filter is differentiable because it is implemented as a `pytensor.scan` loop.

### 8.5 Summary: Kalman Filter vs Hamilton Filter

| Feature | Kalman filter | Hamilton filter |
|---------|-------------|----------------|
| State space | Continuous ($\mathbf{f}_t \in \mathbb{R}^3$) | Discrete ($z_t \in \{0,1\}$) |
| State distribution | Gaussian | Categorical (probabilities) |
| Predict step | Linear propagation via $\boldsymbol{\Phi}$ | Matrix-vector multiply via $P^\top$ |
| Update step | Kalman gain + innovation | Bayes' rule (element-wise multiply + normalise) |
| Likelihood | Gaussian prediction error | Mixture likelihood |
| Implementation | `pytensor.scan` | `pytensor.scan` |